# Fraud Detection - Dataset Loading & Preprocessing

Dataset: [Fraud Detection - 1M Transactions, 7 Fraud Types](https://www.kaggle.com/datasets/sergionefedov/fraud-detection-1m-transactions-7-fraud-types) (Kaggle)

This notebook loads the raw transactions CSV, cleans it, and prepares
train/test splits that are ready to feed into a `scikit-learn` model.

### What is a docstring?

A **docstring** (documentation string) is a string literal written as the
*first statement* inside a function, class, or module. Python stores it in
the object's `__doc__` attribute, and tools like `help()`, IDEs, and
documentation generators (Sphinx, pydoc) read it automatically to show what
the code does.

```python
def add(a, b):
    """Return the sum of a and b."""
    return a + b

print(add.__doc__)  # -> "Return the sum of a and b."
help(add)            # -> shows the same text in an interactive help page
```

Docstrings are different from regular `#` comments:

| | Docstring | Comment |
|---|---|---|
| Syntax | `"""..."""` right after a `def`/`class` | `# ...` anywhere in code |
| Stored at runtime? | Yes, in `__doc__` | No, stripped by the interpreter |
| Purpose | Explains **what** a function/class does, its parameters, and return value, for anyone importing/using it | Explains **why** a specific line of code is written the way it is, for someone reading the source |
| Read by tools | `help()`, IDE tooltips, Sphinx/pydoc | Not accessible programmatically |

Below, every function has a docstring (describing its purpose, parameters
and return value), and `#` comments explain individual non-obvious steps.


## 1. Imports

Only `pandas`, `numpy`, and `scikit-learn` are needed to load and process
the data (as permitted by the assignment).

In [ ]:
import os

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


## 2. Load the dataset

Download the CSV from Kaggle and place it in the same folder as this
notebook (or update `DATA_PATH` below to point at it). The function below
loads the raw file into a `pandas.DataFrame` without modifying it, so the
raw data is always available for inspection.

In [ ]:
DATA_PATH = "fraud_transactions.csv"  # update this to the downloaded CSV's path


def load_fraud_dataset(csv_path: str) -> pd.DataFrame:
    """
    Load the raw fraud-detection transactions dataset from disk.

    Parameters
    ----------
    csv_path : str
        Path to the dataset CSV file downloaded from Kaggle
        (sergionefedov/fraud-detection-1m-transactions-7-fraud-types).

    Returns
    -------
    pandas.DataFrame
        The unmodified transactions data, exactly as read from disk.

    Raises
    ------
    FileNotFoundError
        If no file exists at `csv_path`, with instructions on how to fix it.
    """
    if not os.path.exists(csv_path):
        raise FileNotFoundError(
            f"Could not find '{csv_path}'. Download the dataset from "
            "https://www.kaggle.com/datasets/sergionefedov/fraud-detection-1m-transactions-7-fraud-types "
            "and place the CSV next to this notebook, or update DATA_PATH."
        )
    # low_memory=False avoids dtype-guessing warnings on a wide, mixed-type CSV
    return pd.read_csv(csv_path, low_memory=False)


# Load the data and take a first look at its shape and columns
raw_df = load_fraud_dataset(DATA_PATH)
print(f"Loaded {raw_df.shape[0]:,} rows and {raw_df.shape[1]} columns")
raw_df.head()


## 3. Identify the target column

The dataset labels each transaction with a fraud type (7 fraud types plus
"not fraud"). We auto-detect the most likely target column by name so the
notebook keeps working even if the exact column name differs slightly
between dataset versions; adjust `TARGET_CANDIDATES` if needed.

In [ ]:
TARGET_CANDIDATES = [
    "fraud_type", "isFraud", "is_fraud", "Fraud_Type", "class", "Class", "target",
]


def find_target_column(df: pd.DataFrame, candidates: list) -> str:
    """
    Find which of the candidate column names is present in `df`.

    Parameters
    ----------
    df : pandas.DataFrame
        The dataframe to search for a target/label column.
    candidates : list of str
        Column names to look for, in priority order.

    Returns
    -------
    str
        The first matching column name found in `df.columns`.

    Raises
    ------
    ValueError
        If none of the candidate names are present in `df`.
    """
    for name in candidates:
        if name in df.columns:
            return name
    raise ValueError(
        f"None of {candidates} were found in the dataframe columns: "
        f"{list(df.columns)}. Update TARGET_CANDIDATES with the correct label column."
    )


target_col = find_target_column(raw_df, TARGET_CANDIDATES)
print(f"Using '{target_col}' as the target column")
raw_df[target_col].value_counts()


## 4. Clean and preprocess

The `preprocess_data` function below performs the standard steps needed
before training a `scikit-learn` model:

1. Drop exact duplicate rows.
2. Split features (`X`) from the label (`y`).
3. Impute missing values (median for numeric columns, a placeholder
   category for categorical columns).
4. One-hot encode categorical columns.
5. Scale numeric columns to zero mean / unit variance.
6. Split into train and test sets, stratified by the label so the class
   balance (important for fraud detection, where fraud is rare) is
   preserved in both splits.

In [ ]:
def preprocess_data(
    df: pd.DataFrame,
    target_col: str,
    test_size: float = 0.2,
    random_state: int = 42,
):
    """
    Clean a raw transactions dataframe and split it into train/test sets.

    Parameters
    ----------
    df : pandas.DataFrame
        Raw transactions data, including the target column.
    target_col : str
        Name of the column containing the fraud label.
    test_size : float, default=0.2
        Fraction of rows to hold out for the test set.
    random_state : int, default=42
        Seed used for the train/test split, so results are reproducible.

    Returns
    -------
    X_train, X_test : pandas.DataFrame
        Preprocessed (encoded + scaled) feature matrices.
    y_train, y_test : pandas.Series
        Corresponding fraud labels.
    scaler : sklearn.preprocessing.StandardScaler
        The fitted scaler, kept so new data can be transformed the same way
        before being passed to a trained model.
    """
    # 1. Remove exact duplicate transactions
    df = df.drop_duplicates().reset_index(drop=True)

    # 2. Separate features from the label
    y = df[target_col]
    X = df.drop(columns=[target_col])

    # Drop obvious identifier columns (they don't generalize / would leak row identity)
    id_like_cols = [c for c in X.columns if "id" in c.lower()]
    X = X.drop(columns=id_like_cols, errors="ignore")

    # 3. Identify numeric vs categorical columns for type-appropriate handling
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

    # Impute missing values: median is robust to outliers for numeric columns,
    # while a placeholder string keeps missing categories as their own category
    X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())
    X[categorical_cols] = X[categorical_cols].fillna("Unknown")

    # 4. One-hot encode categorical columns (drop_first avoids redundant columns)
    X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

    # 5. Scale numeric features to zero mean / unit variance
    scaler = StandardScaler()
    X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

    # 6. Stratified split keeps the (rare) fraud classes proportionally
    #    represented in both the train and test sets
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    return X_train, X_test, y_train, y_test, scaler


X_train, X_test, y_train, y_test, scaler = preprocess_data(raw_df, target_col)

print(f"Train set: {X_train.shape}")
print(f"Test set:  {X_test.shape}")
print("\nTrain label distribution:")
print(y_train.value_counts(normalize=True))


`X_train`, `X_test`, `y_train`, and `y_test` are now ready to be passed
directly into any `scikit-learn` classifier (e.g.
`RandomForestClassifier`, `LogisticRegression`, `XGBClassifier`) for
training and evaluation in the next cell.